# 🗺️ Roadmap completo del trabajo de Recomendación

> **Asignatura:** Algoritmos de Recomendación y Computación Social
> **Objetivo:** Comparar ≥4 técnicas de filtrado colaborativo + baseline sobre dataset de música (Million Song Dataset / Spotify + Last.fm).
> **Modelos obligatorios:** Neural Collaborative Filtering (GMF + MLP).
> **Métricas:** MAE, RMSE, Precision@K, Recall@K, F1@K, nDCG@K.

---

## ⚠️ Antes de empezar (sin conexión)

**Asegúrate de tener acordado con tu compañero/a:**
- Valor de `k` para k-core filtering.
- Cómo transformar `playcount` → rating.
- Umbral de "relevante" para métricas de ranking.
- Valor de `K` en top-K (típicamente 10).
- Semilla aleatoria común (ej: 42).
- Reparto de modelos.

**Workflow git:** una rama por persona, merges a `main` solo cuando estable.

---

## FASE 0 — Estado de partida ✅

- Repo clonado, `.venv` activo en VS Code.
- `data_processing.ipynb` con ETL inicial (lo modificaremos en Fase 2).
- EDA empezado con cifras generales:
  - 962,037 usuarios únicos
  - 30,459 canciones únicas
  - 9,711,301 interacciones
  - Densidad: **0.033%**
- Estadísticos de interacciones por usuario:
  - mediana=5, media=10.1, percentil 95=35, percentil 99=71, max=784

---

## FASE 1 — Terminar el EDA (`notebooks/01_eda.ipynb`)

### 1.1 Interacciones por usuario (pendiente terminar)

**Pendiente:**
- [ ] Tabla de % de usuarios con < k interacciones para k ∈ {3, 5, 10, 20, 50}
- [ ] Histograma con escala log en X
- [ ] Markdown con conclusión

**Pista del bucle de %:**
```python
for k in [3, 5, 10, 20, 50]:
    pct = (interaccion_user < k).mean() * 100
    print(f"% usuarios con < {k} interacciones: {pct:.1f}%")
```

**Pista del histograma con escala log:**
```python
import numpy as np
import matplotlib.pyplot as plt
bins = np.logspace(0, np.log10(interaccion_user.max()), 50)
plt.hist(interaccion_user, bins=bins)
plt.xscale('log')
plt.xlabel('Nº interacciones por usuario (log)')
plt.ylabel('Nº usuarios')
plt.title('Distribución de interacciones por usuario')
plt.show()
```

### 1.2 Interacciones por ítem

Replica todo lo de 1.1 pero sobre `track_id`:
```python
interaccion_item = df_history["track_id"].value_counts()
```
Espera distribución aún más extrema (cola larga de canciones raras).

### 1.3 Distribución de playcounts

```python
df_history['playcount'].describe()
df_history['playcount'].quantile([0.9, 0.95, 0.99, 0.999])
```
- Histograma escala log.
- % de interacciones con `playcount == 1`.

### 1.4 Conclusiones del EDA (markdown)

**Decisión A — Valores de k_user y k_item.**
Criterio: el k que conserve entre 30% y 60% de usuarios/ítems.
**Propuesta de base:** `k_user=5, k_item=10`. **Justifícalo con tus números**.

**Decisión B — Transformación de playcount.**
**Recomendación firme:** `log1p` + escalado min-max a [1, 5]:
1. `log_pc = np.log1p(playcount)`
2. `rating = 1 + 4 * (log_pc - log_pc.min()) / (log_pc.max() - log_pc.min())`

Esto encaja con el notebook de clase (escala 1-5, MSELoss). Justificación académica: la transformación log es estándar en literatura de implicit feedback (Hu, Koren & Volinsky 2008).

---

## FASE 2 — ETL definitivo (`notebooks/02_etl_final.ipynb`)

⚠️ **Notebook NUEVO. No toques `data_processing.ipynb` original.**

### 2.1 Cargar CSV crudo
Mismos dtypes que el ETL original (`category`, `uint16`).

### 2.2 K-core iterativo

⚠️ **Es iterativo, no de una pasada.** Al filtrar usuarios cambian los conteos de ítems, y viceversa.

```python
k_user = 5   # ajusta según EDA
k_item = 10  # ajusta según EDA

while True:
    n_antes = len(df)
    # Filtrar usuarios
    counts_u = df['user_id'].value_counts()
    df = df[df['user_id'].isin(counts_u[counts_u >= k_user].index)]
    # Filtrar items
    counts_i = df['track_id'].value_counts()
    df = df[df['track_id'].isin(counts_i[counts_i >= k_item].index)]
    if len(df) == n_antes:
        break
    print(f"Iteración: quedan {len(df)} interacciones")
```

### 2.3 Re-categorizar IDs

⚠️ **Crítico para PyTorch.** Tras filtrar, las categorías mantienen valores "huérfanos". Hay que reindexar a 0..N-1:

```python
df['user_id'] = df['user_id'].astype(str).astype('category')
df['track_id'] = df['track_id'].astype(str).astype('category')
df['user_idx'] = df['user_id'].cat.codes
df['item_idx'] = df['track_id'].cat.codes
```

### 2.4 Transformar playcount → rating

```python
import numpy as np
log_pc = np.log1p(df['playcount'])
df['rating'] = 1 + 4 * (log_pc - log_pc.min()) / (log_pc.max() - log_pc.min())
```

### 2.5 Split estratificado por usuario 80/20

⚠️ **No usar split aleatorio sobre interacciones**. Cada usuario debe aparecer en train y test.

```python
from sklearn.model_selection import train_test_split

# Una opción: groupby + apply (más control)
def split_user(group):
    return train_test_split(group, test_size=0.2, random_state=42)

# Otra opción más rápida con shuffle dentro de cada grupo
# Investiga: df.groupby('user_idx', group_keys=False).apply(...)
```

**Pista alternativa más eficiente:**
```python
# Asignar aleatoriamente cada interacción dentro de su grupo de usuario
np.random.seed(42)
df['random'] = np.random.rand(len(df))
df['rank_in_user'] = df.groupby('user_idx')['random'].rank(pct=True)
train_df = df[df['rank_in_user'] <= 0.8]
test_df = df[df['rank_in_user'] > 0.8]
```

### 2.6 Guardar

```python
from scipy.sparse import csr_matrix, save_npz

num_users = df['user_idx'].nunique()
num_items = df['item_idx'].nunique()

train_set = csr_matrix(
    (train_df['rating'].values, (train_df['user_idx'], train_df['item_idx'])),
    shape=(num_users, num_items)
)
test_set = csr_matrix(
    (test_df['rating'].values, (test_df['user_idx'], test_df['item_idx'])),
    shape=(num_users, num_items)
)

save_npz('../data/processed/train_set.npz', train_set)
save_npz('../data/processed/test_set.npz', test_set)
```

**Guarda también los mapeos** ID original → índice por si necesitas interpretar resultados luego.

---

## FASE 3 — Código compartido (`src/`)

### Estructura
```
src/
  __init__.py        ← vacío
  data_utils.py
  metrics.py
```

### `src/data_utils.py`
Funciones:
- `load_train_test(path)` → devuelve `train_sparse, test_sparse`
- `sparse_to_pairs(matrix)` → arrays `users, items, ratings`

```python
import numpy as np
from scipy.sparse import load_npz

def load_train_test(path='../data/processed/'):
    train = load_npz(path + 'train_set.npz')
    test = load_npz(path + 'test_set.npz')
    return train, test

def sparse_to_pairs(matrix):
    coo = matrix.tocoo()
    return coo.row, coo.col, coo.data
```

### `src/metrics.py`
Funciones:
- `compute_mae(y_true, y_pred)`
- `compute_rmse(y_true, y_pred)`
- `precision_recall_at_k(...)`
- `ndcg_at_k(...)`
- `evaluate_model(...)` → dict con todas

**Importar desde notebook:**
```python
import sys
sys.path.append('..')
from src.metrics import evaluate_model
from src.data_utils import load_train_test
```

---

## FASE 4 — Baseline de la media (`notebooks/03_baseline.ipynb`)

### Concepto
Predecir la media global (o por usuario / por ítem) para cada interacción de test.

### Pasos
```python
# 1. Cargar
train, test = load_train_test()
test_users, test_items, test_ratings = sparse_to_pairs(test)

# 2. Media global de train
train_ratings = train.data
media_global = train_ratings.mean()

# 3. Predecir
y_pred = np.full(len(test_ratings), media_global)

# 4. Evaluar
# MAE, RMSE directos. Ranking: trivialmente bajo (todas las predicciones iguales).
```

**Prueba 3 variantes** para tener material de comparación:
- Media global
- Media por usuario (si usuario nuevo → media global)
- Media por ítem (si ítem nuevo → media global)

---

## FASE 5 — KNN (`notebooks/04_knn.ipynb`)

### Concepto
User-based KNN con similitud coseno.

### Pasos
1. Calcular matriz de similitud usuario-usuario:
```python
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(train.astype(np.float32))
```
2. Para cada `(u, i)` de test: top-K vecinos de `u` que tengan rating en `i`, media ponderada por similitud.
3. Evaluar.

**Hiperparámetro:** K ∈ {10, 20, 50, 100}.

⚠️ **Cuidado con la RAM:** con 50k usuarios, la matriz de similitud son 50k×50k floats ≈ 10GB en float32. Si peta, considera:
- Usar `sparse` (similitud sparse con `linear_kernel`).
- Calcular similitudes on-the-fly por bloques.
- Usar `NearestNeighbors` de sklearn que mantiene solo los top-K.

---

## FASE 6 — PMF con bias (`notebooks/05_pmf.ipynb`)

### Concepto
$\hat{r}_{ui} = \mu + b_u + b_i + p_u \cdot q_i$

### Opción rápida — librería surprise
```python
from surprise import SVD, Dataset, Reader

reader = Reader(rating_scale=(1, 5))
# ... cargar datos en formato surprise
model = SVD(n_factors=32, n_epochs=20, lr_all=0.005, reg_all=0.02)
model.fit(trainset)
```

⚠️ Si usas surprise, **recalcula las métricas con tu `metrics.py`** para que la comparación sea justa.

### Opción manual — PyTorch
Idéntico a GMF pero sin la capa Linear final (producto escalar puro + bias).

**Hiperparámetros:** `n_factors`, `n_epochs`, `lr`, `reg`.

---

## FASE 7 — GMF (`notebooks/06_gmf.ipynb`) ⭐ TU PARTE PRINCIPAL

### 7.1 Modelo

```python
import torch
import torch.nn as nn

class GMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, latent_dim)
        self.item_emb = nn.Embedding(num_items, latent_dim)
        self.output = nn.Linear(latent_dim, 1)

    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)        # (batch, latent_dim)
        i = self.item_emb(item_ids)        # (batch, latent_dim)
        prod = u * i                       # producto Hadamard (elemento a elemento)
        return self.output(prod).squeeze() # (batch,)
```

**Idea clave:** `u * i` es producto elemento a elemento. La capa Linear final generaliza el producto escalar puro de PMF.

### 7.2 Datos como tensores

```python
from torch.utils.data import TensorDataset, DataLoader

user_tensor = torch.LongTensor(train_users)
item_tensor = torch.LongTensor(train_items)
rating_tensor = torch.FloatTensor(train_ratings)

dataset = TensorDataset(user_tensor, item_tensor, rating_tensor)
loader = DataLoader(dataset, batch_size=1024, shuffle=True)
```

### 7.3 Entrenamiento

```python
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GMF(num_users, num_items, latent_dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for users, items, ratings in loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss = loss_fn(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}: loss = {avg_loss:.4f}")
```

### 7.4 Predicción

```python
model.eval()
with torch.no_grad():
    test_u = torch.LongTensor(test_users).to(device)
    test_i = torch.LongTensor(test_items).to(device)
    y_pred = model(test_u, test_i).cpu().numpy()
```

### 7.5 Búsqueda de hiperparámetros ⭐ (MUY VALORADO)

**Estrategia:** varía UNO a la vez, fija el resto en valores razonables.

| Hiperparámetro | Valores a probar | Default si fijo |
|----------------|------------------|------------------|
| `latent_dim` | 8, 16, 32, 64 | 32 |
| `learning_rate` | 0.01, 0.001, 0.0001 | 0.001 |
| `num_epochs` | 5, 10, 20, 50 | 20 |
| `batch_size` | 256, 1024, 4096 | 1024 |

Guarda resultados en tabla pandas:
```python
results = []
for ld in [8, 16, 32, 64]:
    # ... entrenar ...
    results.append({
        'latent_dim': ld,
        'mae': ..., 'rmse': ..., 'precision': ..., etc.
    })
df_results = pd.DataFrame(results)
```

**Gráficas obligatorias para la memoria:**
- Loss por epoch (training curve).
- RMSE vs latent_dim.
- RMSE vs learning_rate.
- Métricas vs epochs (¿overfitting?).

---

## FASE 8 — MLP (`notebooks/07_mlp.ipynb`) ⭐ TU PARTE PRINCIPAL

### 8.1 Modelo

```python
class MLP(nn.Module):
    def __init__(self, num_users, num_items, latent_dim, hidden_layers):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, latent_dim)
        self.item_emb = nn.Embedding(num_items, latent_dim)

        layers = []
        input_size = latent_dim * 2  # CONCATENACIÓN
        for h in hidden_layers:
            layers.append(nn.Linear(input_size, h))
            layers.append(nn.ReLU())
            input_size = h
        layers.append(nn.Linear(input_size, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)
        i = self.item_emb(item_ids)
        x = torch.cat([u, i], dim=1)  # ¡concatenar, no multiplicar!
        return self.mlp(x).squeeze()
```

⚠️ **Diferencia crítica con GMF:** aquí concatenas (`torch.cat`) en vez de multiplicar. Las capas densas no-lineales aprenden la interacción.

### 8.2 Búsqueda de hiperparámetros (extra respecto a GMF)

Arquitecturas a probar (típico patrón "torre" decreciente):
- `[64, 32]`
- `[128, 64, 32]`
- `[256, 128, 64, 32]`

Resto igual que GMF.

**Opcional:** prueba añadir Dropout:
```python
layers.append(nn.Dropout(0.2))
```

### 8.3 Análisis ⭐
Pregunta clave del trabajo: **¿GMF o MLP?**
- A veces gana GMF (interacciones lineales suficientes).
- A veces gana MLP (patrones no lineales relevantes).
- A veces ganan en métricas distintas.

Esto va a las conclusiones, comparándolo con lo que dice He et al. (2017).

---

## FASE 9 — Comparativa final (`notebooks/08_comparativa.ipynb`)

### 9.1 Tabla resumen

| Modelo | MAE | RMSE | Precision@10 | Recall@10 | F1@10 | nDCG@10 | Tiempo (s) |
|--------|-----|------|--------------|-----------|-------|---------|------------|
| Baseline media | ... | ... | ... | ... | ... | ... | ... |
| KNN (k=50) | ... | ... | ... | ... | ... | ... | ... |
| PMF (d=32) | ... | ... | ... | ... | ... | ... | ... |
| GMF (d=32) | ... | ... | ... | ... | ... | ... | ... |
| MLP ([64,32]) | ... | ... | ... | ... | ... | ... | ... |

### 9.2 Gráficas

- Bar chart agrupado: cada modelo, cada métrica.
- Separar métricas de error (MAE, RMSE) de las de ranking (escalas distintas).
- Scatter "tiempo vs métrica" (frontera de Pareto).

### 9.3 Análisis (markdown)

Preguntas a contestar:
- ¿Qué modelo gana en cada métrica?
- ¿Coincide el ganador en MAE/RMSE con el ganador en Precision/nDCG?
- ¿Cuánto mejoran respecto al baseline?
- GMF vs MLP, ¿quién gana? ¿Coherente con el paper original?
- Trade-off tiempo de entrenamiento / ganancia.

---

## FASE 10 — Memoria/Informe

### Estructura recomendada

1. **Introducción** (1 párrafo). Qué es filtrado colaborativo, qué se compara.
2. **Dataset.** Origen, tamaño, decisiones del EDA, k-core, transformación, split.
3. **Modelos.** Sección breve por cada uno: idea, fórmula, hiperparámetros.
4. **Evaluación.** Definición exacta de cada métrica. Sobre todo: cómo defines "relevante".
5. **Resultados.** Tablas, gráficos de búsqueda de hiperparámetros, tiempos.
6. **Análisis.** Lo de la fase 9.
7. **Conclusiones.** Aprendizajes, limitaciones, trabajo futuro.

---

## 🛠️ Métricas de ranking — el detalle complicado

### Cómo se calculan Precision@K, Recall@K, nDCG@K

Para cada usuario en test:

1. **Predecir rating** para todos los items que tiene en test (o ideal: todos los items).
2. **Ordenar** items por predicción descendente.
3. **Coger top-K**.
4. **Definir "relevante":** items con rating real ≥ umbral (recomendación: ≥ 4).
5. **Precision@K** = (relevantes en top-K) / K.
6. **Recall@K** = (relevantes en top-K) / (total relevantes del usuario en test).
7. **F1@K** = 2·P·R / (P+R).
8. **nDCG@K:**
   - `DCG@K = Σ (2^rel_i - 1) / log2(i + 1)` sobre top-K
   - `IDCG@K` = DCG del ranking ideal
   - `nDCG@K = DCG / IDCG`

Promediar sobre todos los usuarios.

### Esqueleto de implementación

```python
def precision_recall_at_k(user_predictions, test_dict, k=10, threshold=4.0):
    """
    user_predictions: dict {user: [(item, pred_rating), ...]}
    test_dict: dict {user: [(item, true_rating), ...]}
    """
    precisions, recalls = [], []
    for user, preds in user_predictions.items():
        # Ordenar por predicción descendente
        preds_sorted = sorted(preds, key=lambda x: x[1], reverse=True)
        top_k_items = [item for item, _ in preds_sorted[:k]]

        # Verdaderos relevantes del usuario en test
        true_relevant = {item for item, r in test_dict[user] if r >= threshold}

        if not true_relevant:
            continue  # saltar usuarios sin relevantes

        hits = sum(1 for item in top_k_items if item in true_relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(true_relevant))

    return np.mean(precisions), np.mean(recalls)
```

⚠️ **Decisión a tomar y dejar escrita en la memoria:**
- Umbral de "relevante": ¿`≥ 4.0`? ¿Percentil 75?
- ¿Qué hacer con usuarios sin relevantes? (Saltar / contar 0).

---

## 🚨 Trampas habituales

1. **Reproducibilidad:** semillas fijas en todo: `torch.manual_seed(42)`, `np.random.seed(42)`.

2. **Tiempos:** GMF/MLP en CPU con 50k usuarios + 30 epochs ≈ 10-30 min por configuración. Si tienes GPU: `model.to('cuda')` y los tensores también.

3. **Overfitting:** loss baja en train pero RMSE sube en test → menos epochs, weight_decay, dropout.

4. **Métricas raras:**
   - Recall > 1 → bug.
   - nDCG > 1 → bug.
   - Si usuarios tienen < K items en test, métricas se rompen → por eso k-core.

5. **IDs categóricos:** tras filtrar siempre re-categorizar a 0..N-1, o PyTorch peta con IndexError.

6. **Embeddings huge:** si num_users > 100k, latent_dim ≤ 32 para que entre en RAM/GPU.

7. **Backup:** commitea regularmente. Si trabajáis dos, ramas separadas.

---

## 📋 Checklist final antes de entregar

- [ ] EDA con justificación de filtrado.
- [ ] ETL definitivo aplicando decisiones del EDA.
- [ ] Baseline implementado y evaluado.
- [ ] Al menos 4 técnicas + baseline.
- [ ] NCF (GMF + MLP) implementadas en PyTorch.
- [ ] Búsqueda de hiperparámetros documentada (al menos en GMF y MLP).
- [ ] Las 6 métricas calculadas para todos los modelos.
- [ ] Tabla comparativa y gráficos finales.
- [ ] Memoria con introducción, metodología, resultados, análisis y conclusiones.
- [ ] Reproducibilidad: semillas fijas, código en `src/`, README explicando cómo ejecutar.
- [ ] Commits en git, repo limpio.

---

## 📚 Referencias útiles

- **He et al. (2017)** — "Neural Collaborative Filtering". WWW 2017. Paper original de NCF.
- **Hu, Koren & Volinsky (2008)** — "Collaborative Filtering for Implicit Feedback Datasets". Justifica el uso de log y weighted approaches.
- **Documentación PyTorch:** `nn.Embedding`, `TensorDataset`, `DataLoader`.
- **Documentación scikit-surprise:** `SVD`, `KNNBasic`.

---

## 💡 Si te atascas sin conexión

- **No avanza el entrenamiento:** baja `batch_size`, sube `lr`, comprueba que los IDs estén en 0..N-1.
- **CUDA out of memory:** baja `batch_size` o `latent_dim`. Pasa a CPU.
- **Métricas absurdas:** revisa que train y test no se solapen, que los IDs estén bien mapeados, que el umbral de relevante sea coherente con tu escala.
- **PyTorch IndexError en embedding:** seguro que tienes un user_id ≥ num_users. Re-categoriza.

---

¡Mucho ánimo! Cuando vuelvas con conexión, retomamos donde lo dejaste.
